# Coarse 1 km WRF vs FuXi-CFD 30 m downscaling

Compares the existing 1 km WRG deliverables against the FuXi-CFD high-resolution
WRG at the same hub heights (80 / 100 / 120 / 160 m).

**The question this notebook answers is not "which is right".** WRF at 1 km is the
*driving* data — FuXi-CFD is given its 100 m winds and asked to add sub-kilometre
structure. So two things matter, and they are checked separately:

1. **Agreement at the coarse scale.** Averaged back up to 1 km, the high-res field
   should reproduce WRF. Large bias here means something is wrong in the chain,
   not that we have learned something about the wind.
2. **Structure below 1 km.** The spread *within* each 1 km cell is the entire
   product. If it is near zero, FuXi-CFD added nothing at this site — a real
   possibility given Broadview's 49–228 m of relief per tile.

### Power density is recomputed on both sides

Different producers write different quantities into the WRG power-density column
(turbine kW vs W/m²), so that column is ignored. Both sides get
`0.5·ρ·A³·Γ(1+3/k)` — the analytic W/m² for a Weibull — via `wrg_read`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fuxicfd_wrf.wrg_read import read_wrg, to_grid, weibull_mean_speed

COARSE_DIR = Path("/scratch/moptis/c2wind/broadview/control/deliverables/wrg/broadview")
FINE_DIR = Path("/scratch/moptis/fuxicfd/out/wrg")
HEIGHTS = [80, 100, 120, 160]
AIR_DENSITY = 1.06  # Broadview sits at ~1340 m; 1.225 is sea level

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

In [ ]:
coarse, fine = {}, {}
for h in HEIGHTS:
    cp = COARSE_DIR / f"control_{h}m.wrg"
    fp = FINE_DIR / f"broadview_{h}m.wrg"
    if cp.exists():
        coarse[h] = read_wrg(cp, air_density=AIR_DENSITY)
    if fp.exists():
        fine[h] = read_wrg(fp, air_density=AIR_DENSITY)

def _desc(df):
    if df is None:
        return 'missing'
    a = df.attrs
    return f"{len(df):>8,} pts @ {a['cell_size']:.0f} m, {a['n_sectors']} sectors"

for h in HEIGHTS:
    print(f"{h:4d} m | coarse {_desc(coarse.get(h))} | fine {_desc(fine.get(h))}")

## 1. Aggregate the high-res field back to the coarse grid

Each coarse cell is matched to the fine points falling inside it. This is the
like-for-like comparison; anything else compares different spatial scales.

In [ ]:
def aggregate_to_coarse(fine_df, coarse_df):
    """Bin fine points into coarse cells; return per-cell mean and within-cell std."""
    cell = coarse_df.attrs["cell_size"]
    cx = np.sort(coarse_df["x"].unique())
    cy = np.sort(coarse_df["y"].unique())
    # Snap each fine point to its enclosing coarse cell centre.
    ix = np.round((fine_df["x"].values - cx[0]) / cell).astype(int)
    iy = np.round((fine_df["y"].values - cy[0]) / cell).astype(int)
    ok = (ix >= 0) & (ix < len(cx)) & (iy >= 0) & (iy < len(cy))

    g = pd.DataFrame({
        "x": cx[ix[ok]], "y": cy[iy[ok]],
        "mean_speed": fine_df["mean_speed"].values[ok],
        "power_density": fine_df["power_density"].values[ok],
        "A": fine_df["A"].values[ok], "k": fine_df["k"].values[ok],
    })
    agg = g.groupby(["x", "y"]).agg(
        fine_speed=("mean_speed", "mean"),
        fine_speed_std=("mean_speed", "std"),
        fine_speed_min=("mean_speed", "min"),
        fine_speed_max=("mean_speed", "max"),
        fine_pd=("power_density", "mean"),
        fine_pd_std=("power_density", "std"),
        fine_A=("A", "mean"), fine_k=("k", "mean"),
        n_fine=("mean_speed", "size"),
    ).reset_index()
    return coarse_df.merge(agg, on=["x", "y"], how="inner")


merged = {h: aggregate_to_coarse(fine[h], coarse[h]) for h in HEIGHTS if h in fine and h in coarse}
for h, m in merged.items():
    print(f"{h} m: {len(m):,} coarse cells matched, median {m['n_fine'].median():.0f} fine points per cell")

## 2. Does the high-res field reproduce WRF at 1 km?

A bias of a few percent is expected — FuXi-CFD has its own error floor (RMSE
0.15 m/s against the CFD it was trained on) and Broadview sits ~4σ outside its
training distribution. A bias of tens of percent means the chain is broken.

In [ ]:
rows = []
for h, m in merged.items():
    d = m["fine_speed"] - m["mean_speed"]
    dp = m["fine_pd"] - m["power_density"]
    rows.append({
        "height_m": h,
        "coarse_speed": m["mean_speed"].mean(),
        "fine_speed": m["fine_speed"].mean(),
        "bias_ms": d.mean(),
        "bias_pct": 100 * d.mean() / m["mean_speed"].mean(),
        "rmse_ms": np.sqrt((d**2).mean()),
        "corr": m["fine_speed"].corr(m["mean_speed"]),
        "coarse_pd": m["power_density"].mean(),
        "fine_pd": m["fine_pd"].mean(),
        "pd_bias_pct": 100 * dp.mean() / m["power_density"].mean(),
    })
summary = pd.DataFrame(rows).set_index("height_m").round(3)
summary

In [ ]:
fig, axes = plt.subplots(1, len(merged), figsize=(3.4 * len(merged), 3.4), squeeze=False)
for ax, (h, m) in zip(axes[0], merged.items()):
    ax.scatter(m["mean_speed"], m["fine_speed"], s=3, alpha=0.25, lw=0)
    lim = [min(m["mean_speed"].min(), m["fine_speed"].min()),
           max(m["mean_speed"].max(), m["fine_speed"].max())]
    ax.plot(lim, lim, "k--", lw=0.8, label="1:1")
    ax.set_xlabel("WRF 1 km mean speed (m/s)")
    ax.set_title(f"{h} m")
    ax.legend(fontsize=7)
axes[0][0].set_ylabel("FuXi-CFD, cell-averaged (m/s)")
fig.suptitle("Coarse-scale agreement: high-res averaged back to 1 km", y=1.02)
fig.tight_layout()

## 3. The actual product: structure *within* each 1 km cell

This is what the 1 km field cannot represent and what the downscaling exists to
produce. If the within-cell spread is negligible, FuXi-CFD is adding nothing here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for h, m in merged.items():
    axes[0].hist(m["fine_speed_std"], bins=60, histtype="step", label=f"{h} m")
    axes[1].hist(m["fine_speed_max"] - m["fine_speed_min"], bins=60, histtype="step", label=f"{h} m")
axes[0].set_xlabel("within-cell std of mean speed (m/s)")
axes[1].set_xlabel("within-cell range, max - min (m/s)")
for a in axes:
    a.set_ylabel("coarse cells")
    a.legend(fontsize=8)
fig.suptitle("Sub-kilometre structure added by FuXi-CFD", y=1.02)
fig.tight_layout()

print("Median within-cell spread (m/s) and as % of cell mean:")
for h, m in merged.items():
    s = m["fine_speed_std"].median()
    print(f"  {h:4d} m : {s:.3f} m/s  ({100 * s / m['fine_speed'].mean():.1f}%)"
          f"   range {(m['fine_speed_max'] - m['fine_speed_min']).median():.3f} m/s")

## 4. Maps side by side

Same colour scale, so the eye compares fairly.

In [ ]:
H = 100 if 100 in fine else sorted(fine)[0]
cx, cy, cg = to_grid(coarse[H], "mean_speed")
fx, fy, fg = to_grid(fine[H], "mean_speed")
vmin, vmax = np.nanpercentile(np.concatenate([cg.ravel(), fg.ravel()]), [2, 98])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, (x, y, g, name) in zip(axes, [
    (cx, cy, cg, f"WRF 1 km — {H} m"),
    (fx, fy, fg, f"FuXi-CFD {fine[H].attrs['cell_size']:.0f} m — {H} m"),
]):
    im = ax.pcolormesh(x / 1000, y / 1000, g, vmin=vmin, vmax=vmax, cmap="viridis", shading="auto")
    ax.set_title(name)
    ax.set_xlabel("UTM 13N easting (km)")
    ax.set_aspect("equal")
    fig.colorbar(im, ax=ax, label="mean speed (m/s)")
axes[0].set_ylabel("northing (km)")
fig.tight_layout()

## 5. Shear between the four heights

The power-law exponent α from a fit across all available heights. Shear is where
a downscaling that resolves terrain should differ most from a 1 km field.

In [ ]:
def shear_exponent(dfs, heights):
    """Least-squares alpha in log(U) = alpha*log(z) + c, per grid point."""
    z = np.log(np.asarray(heights, dtype=float))
    U = np.log(np.stack([dfs[h]["mean_speed"].values for h in heights]))
    return np.polyfit(z, U, 1)[0]

hs = [h for h in HEIGHTS if h in coarse and h in fine]
if len(hs) >= 2:
    a_c = shear_exponent(coarse, hs)
    a_f = shear_exponent(fine, hs)
    fig, ax = plt.subplots(figsize=(5, 3.4))
    ax.hist(a_c, bins=60, histtype="step", label=f"WRF 1 km (median {np.median(a_c):.3f})")
    ax.hist(a_f, bins=60, histtype="step", label=f"FuXi-CFD (median {np.median(a_f):.3f})")
    ax.set_xlabel(r"shear exponent $\alpha$")
    ax.set_ylabel("grid points")
    ax.legend(fontsize=8)
    fig.tight_layout()
else:
    print("need at least two matching heights")

## 6. Sector-wise frequency

Only valid if both files use the same sector count. The coarse Broadview files
use 16 sectors (22.5°), so run the export with `--bin-size 22.5` to compare.

In [ ]:
c, f = coarse[H], fine[H]
if c.attrs["n_sectors"] != f.attrs["n_sectors"]:
    print(
        f"Sector counts differ (coarse {c.attrs['n_sectors']}, fine {f.attrs['n_sectors']})."
        f"\nRe-run the export with --bin-size {360 / c.attrs['n_sectors']:g} to compare directly."
    )
else:
    n = c.attrs["n_sectors"]
    ang = np.arange(n) * 360 / n
    fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(4.6, 4.6))
    for df, name in [(c, "WRF 1 km"), (f, "FuXi-CFD")]:
        freq = [df[f"freq_{i}"].mean() for i in range(n)]
        th = np.deg2rad(np.append(ang, ang[0]))
        ax.plot(th, np.append(freq, freq[0]), label=name)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(f"Directional frequency at {H} m")
    ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1), fontsize=8)